In [1]:
from dataclasses import dataclass
import os
from groq import Groq
@dataclass(frozen=True)
class Provider:
    """One provider to reliably route requests across all inference providers."""
    
    name: str
    env_vars: str
    is_free: bool
    base_url: str | None
    model: str
    
PROVIDERS = {
    "groq": Provider(
        name="Groq",
        env_vars="GROQ_API_KEY",
        is_free=False,
        base_url="https://api.groq.com/openai/v1",
        model="openai/gpt-oss-120b"  
    ),
    # can i use one more geoq provider -> answer: yes, you can add another provider by creating a new instance of the Provider class and adding it to the PROVIDERS dictionary. For example:
    "groq2": Provider(
        name="Groq2",
        env_vars="GROQ2_API_KEY",
        is_free=False,
        base_url="https://api.groq.com/openai/v1",
        model="openai/gpt-oss-20b"  
    ) 
    
}

def select_provider() -> Provider:
    """Select a provider based on the environment variables."""
    for provider in PROVIDERS.values():
        if provider.env_vars in os.environ:
            return provider
        
    expected = ", ".join([provider.env_vars for provider in PROVIDERS.values()])
    raise ValueError(f"No valid provider found. Please set one of the following environment variables: {expected}.")

def build_client(provider: Provider)-> Groq:
    # if provider.base_url is None:
    #     return Groq(api_key=os.environ[provider.env_vars])
    
    # else :
    #     return Groq(api_key=os.environ[provider.env_vars], base_url=provider.base_url)
    return Groq(api_key=os.environ[provider.env_vars])
    
def have_any_key()-> bool:
    return any(os.getenv(provider.env_vars) for provider in PROVIDERS.values())

print("Found a provider with a valid API key." if have_any_key() else "No provider with a valid API key found.")

Found a provider with a valid API key.


In [2]:
def llm_reply(prompt:str)->str:
    provider = select_provider()
    client = build_client(provider)
    response = client.chat.completions.create(
        model=provider.model,
        max_tokens=200,
        messages=[
            {
                "role":"user",
                "content":prompt
            }
        ]
    )
    print(f"Using provider: {provider.name}")
    return response.choices[0].message.content

In [5]:
prompt = "Who was the PM of UK before 2020?"
try:
    print(llm_reply(prompt))
    print("\nSuccessfully got a response from the LLM.")
except Exception as e:
    print(f"Error: {e}")

Using provider: Groq
Before 2020 the United Kingdom had two different Prime Ministers:

| Period | Prime Minister | Party |
|--------|----------------|-------|
| 13 July 2016 – 24 July 2019 | **Theresa May** | Conservative |
| 24 July 2019 – 13 December 2022 | **Boris Johnson** | Conservative |

Theresa

Successfully got a response from the LLM.
